In [1]:
import csv, json, os, warnings, numpy as np
from tslearn.clustering import KShape
from pyspark.sql import SparkSession, functions as F, types as T

warnings.filterwarnings("ignore")


class SafeKShapeClustering:
    def __init__(self):
        self.cluster_input_path = "/user/data/kshape/preclustering"
        self.fallback_model_ready_path = "/user/data/kshape/preprocess/model_ready"
        self.full_panel_path = "/user/data/kshape/preprocess/full_panel"
        self.assignments_csv_out = "/workspace/code/kshape/clustering/assignments.csv"
        self.cluster_meta_dir = "/workspace/code/kshape/clustering/meta"
        self.panel_with_cluster_out = "/user/data/kshape/clustering"

        self.n_clusters, self.random_state, self.n_init = 3, 42, 1
        self.timezone = "America/New_York"
        self.max_driver_matrix_cells, self.max_time_bins, self.max_kshape_work_gb = 12_000_000, 4096, 4.0
        self.auto_resample_for_safety, self.min_bucket_minutes = True, 30

        os.makedirs(self.cluster_meta_dir, exist_ok=True)
        os.makedirs(os.path.dirname(self.assignments_csv_out), exist_ok=True)

        self.spark = (
            SparkSession.builder.appName("Clustering")
            .config("spark.driver.maxResultSize", "1g")
            .config("spark.sql.session.timeZone", self.timezone)
            .config("spark.sql.files.ignoreCorruptFiles", "true")
            .config("spark.sql.parquet.mergeSchema", "true")
            .config("spark.sql.adaptive.enabled", "true")
            .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
            .config("spark.sql.adaptive.skewJoin.enabled", "true")
            .config("spark.sql.shuffle.partitions", "64")
            .config("spark.sql.files.maxPartitionBytes", str(64 * 1024 * 1024))
            .config("spark.sql.debug.maxToStringFields", "500")
            .config("spark.sql.maxPlanStringLength", "50000")
            .config("spark.master", "yarn")
            .config("spark.submit.deployMode", "client")
            .config("spark.driver.host", "192.168.1.111")
            .config("spark.driver.bindAddress", "0.0.0.0")
            .getOrCreate()
        )
        self.spark.sparkContext.setLogLevel("WARN")
        jvm = self.spark._jvm
        self.fs = jvm.org.apache.hadoop.fs.FileSystem.get(self.spark._jsc.hadoopConfiguration())
        self.Path = jvm.org.apache.hadoop.fs.Path

    def exists(self, path):
        return self.fs.exists(self.Path(path))

    def run(self):
        if self.exists(self.cluster_input_path):
            raw_df, cluster_input_source = self.spark.read.parquet(self.cluster_input_path), self.cluster_input_path
        else:
            if not self.exists(self.fallback_model_ready_path):
                raise ValueError(f"Không tìm thấy cả {self.cluster_input_path} và {self.fallback_model_ready_path}.")
            raw_df, cluster_input_source = self.spark.read.parquet(self.fallback_model_ready_path), self.fallback_model_ready_path
            split_col = "dataset_split" if "dataset_split" in raw_df.columns else ("split" if "split" in raw_df.columns else None)
            if split_col:
                raw_df = raw_df.filter(F.col(split_col) == "train")
            raw_df = raw_df.select(
                F.col("pickup_bin_30m").cast("timestamp").alias("pickup_bin_30m"),
                F.col("PULocationID").cast("int").alias("PULocationID"),
                F.coalesce(F.col("pickup_demand"), F.lit(0.0)).cast("double").alias("pickup_demand"),
            )

        missing = sorted({"pickup_bin_30m", "PULocationID", "pickup_demand"} - set(raw_df.columns))
        if missing:
            raise ValueError(f"Thiếu cột bắt buộc cho clustering: {missing}")

        df = (
            raw_df.select(
                F.col("pickup_bin_30m").cast("timestamp").alias("pickup_bin_30m"),
                F.col("PULocationID").cast("int").alias("PULocationID"),
                F.coalesce(F.col("pickup_demand"), F.lit(0.0)).cast("double").alias("pickup_demand"),
            )
            .filter(F.col("pickup_bin_30m").isNotNull() & F.col("PULocationID").isNotNull())
            .groupBy("PULocationID", "pickup_bin_30m")
            .agg(F.sum("pickup_demand").alias("pickup_demand"))
        )

        prev_ts, min_diff_sec = None, None
        for r in df.select("pickup_bin_30m").distinct().orderBy("pickup_bin_30m").toLocalIterator():
            cur = r["pickup_bin_30m"]
            if cur and prev_ts:
                d = int((cur - prev_ts).total_seconds())
                if d > 0 and (min_diff_sec is None or d < min_diff_sec):
                    min_diff_sec = d
            if cur:
                prev_ts = cur

        base_bin_minutes = max(self.min_bucket_minutes, int((min_diff_sec or self.min_bucket_minutes * 60) // 60))
        effective_bin_minutes, was_resampled = base_bin_minutes, False
        n_locations = df.select("PULocationID").distinct().count()
        n_time_bins = df.select("pickup_bin_30m").distinct().count()
        if n_locations == 0: raise ValueError("Không có location nào để clustering.")
        if n_time_bins == 0: raise ValueError("Không có time bin nào để clustering.")
        if n_locations < self.n_clusters: raise ValueError(f"Số location ({n_locations}) nhỏ hơn n_clusters={self.n_clusters}.")

        matrix_cells = n_locations * n_time_bins
        estimated_kshape_work_gb = (n_time_bins * n_time_bins * 8) / (1024 ** 3)
        too_large = (
            matrix_cells > self.max_driver_matrix_cells
            or n_time_bins > self.max_time_bins
            or estimated_kshape_work_gb > self.max_kshape_work_gb
        )

        if too_large:
            if not self.auto_resample_for_safety:
                raise MemoryError(
                    "Chuỗi thời gian quá dài cho K-Shape an toàn. "
                    f"n_locations={n_locations}, n_time_bins={n_time_bins}, matrix_cells={matrix_cells:,}, "
                    f"estimated_kshape_work_gb={estimated_kshape_work_gb:.2f}. Hãy tăng bucket thời gian hoặc giảm time range."
                )
            bucket_minutes = base_bin_minutes
            while True:
                bucket_minutes += base_bin_minutes
                bucket_seconds = bucket_minutes * 60
                candidate = (
                    df.withColumn(
                        "pickup_bin_bucket",
                        F.from_unixtime(
                            (F.unix_timestamp("pickup_bin_30m") / F.lit(bucket_seconds)).cast("bigint") * F.lit(bucket_seconds)
                        ).cast("timestamp")
                    )
                    .groupBy("PULocationID", "pickup_bin_bucket").agg(F.sum("pickup_demand").alias("pickup_demand"))
                    .select("PULocationID", F.col("pickup_bin_bucket").alias("pickup_bin_30m"), "pickup_demand")
                )
                candidate_bins = candidate.select("pickup_bin_30m").distinct().count()
                candidate_cells = n_locations * candidate_bins
                candidate_work_gb = (candidate_bins * candidate_bins * 8) / (1024 ** 3)
                if (
                    candidate_bins <= self.max_time_bins
                    and candidate_cells <= self.max_driver_matrix_cells
                    and candidate_work_gb <= self.max_kshape_work_gb
                ):
                    df, n_time_bins, matrix_cells, estimated_kshape_work_gb = (
                        candidate, candidate_bins, candidate_cells, candidate_work_gb
                    )
                    effective_bin_minutes, was_resampled = bucket_minutes, True
                    break

        time_index = [r["pickup_bin_30m"] for r in df.select("pickup_bin_30m").distinct().orderBy("pickup_bin_30m").collect()]
        if not time_index:
            raise ValueError("time_index rỗng, không thể tạo ma trận K-Shape.")
        time_pos = {ts: i for i, ts in enumerate(time_index)}

        loc_ids, matrix = [], []
        for row in (
            df.groupBy("PULocationID")
            .agg(F.array_sort(F.collect_list(F.struct("pickup_bin_30m", "pickup_demand"))).alias("series"))
            .orderBy("PULocationID").toLocalIterator()
        ):
            values = np.zeros(len(time_index), dtype=np.float64)
            for item in row["series"]:
                values[time_pos[item["pickup_bin_30m"]]] = float(item["pickup_demand"] or 0.0)
            loc_ids.append(int(row["PULocationID"]))
            matrix.append(values)

        if not matrix:
            raise ValueError("Không dựng được ma trận đầu vào cho K-Shape.")

        X = np.stack(matrix)
        Xz = np.stack([((x - np.nanmean(x)) / max(np.nanstd(x), 1e-12)) for x in X])[:, :, None]
        model = KShape(n_clusters=self.n_clusters, random_state=self.random_state, n_init=self.n_init, verbose=False)
        labels = model.fit_predict(Xz).astype(int)
        centers = np.asarray(model.cluster_centers_).squeeze(-1)

        assignments = [(int(i), int(c)) for i, c in zip(loc_ids, labels)]
        with open(self.assignments_csv_out, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["PULocationID", "cluster_id"])
            w.writerows(assignments)
        np.save(os.path.join(self.cluster_meta_dir, "kshape_centroids.npy"), centers)

        if not self.exists(self.full_panel_path):
            raise ValueError(f"Không tìm thấy full_panel_path: {self.full_panel_path}")

        full_panel = self.spark.read.parquet(self.full_panel_path)
        assignments_df = self.spark.createDataFrame(assignments, T.StructType([
            T.StructField("PULocationID", T.IntegerType(), False),
            T.StructField("cluster_id", T.IntegerType(), False),
        ]))
        result = (
            full_panel.join(F.broadcast(assignments_df), on="PULocationID", how="left")
            .withColumn("cluster_id", F.col("cluster_id").cast("int"))
        )

        writer = result.write.mode("overwrite")
        parts = [c for c in ["dataset_split", "month"] if c in result.columns]
        if parts:
            writer = writer.partitionBy(*parts)
        writer.parquet(self.panel_with_cluster_out)

        metadata = {
            "cluster_input_source": cluster_input_source,
            "full_panel_path": self.full_panel_path,
            "assignments_csv_out": self.assignments_csv_out,
            "panel_with_cluster_out": self.panel_with_cluster_out,
            "matrix_shape": [int(X.shape[0]), int(X.shape[1])],
            "n_clusters": self.n_clusters,
            "random_state": self.random_state,
            "n_locations": n_locations,
            "n_time_bins": n_time_bins,
            "matrix_cells": matrix_cells,
            "base_bin_minutes": base_bin_minutes,
            "effective_bin_minutes": effective_bin_minutes,
            "estimated_kshape_work_gb": round(float(estimated_kshape_work_gb), 4),
            "was_resampled": was_resampled,
        }
        with open(os.path.join(self.cluster_meta_dir, "kshape_metadata.json"), "w", encoding="utf-8") as f:
            json.dump(metadata, f, indent=2, ensure_ascii=False)

        print("=" * 120)
        print("STEP 3 — SAFE K-SHAPE CLUSTERING DONE")
        print("=" * 120)
        for k, v in metadata.items():
            print(f"{k}: {v}")
        print("\nCluster counts:")
        for cluster_id, count in zip(*np.unique(labels, return_counts=True)):
            print(f"{int(cluster_id)}: {int(count)}")

        return metadata


SafeKShapeClustering().run()


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b10589c1-775c-4bd7-b7f2-23eed77a6ca2;1.0
	confs: [default]
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 104ms :: artifacts dl 4ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	-----------------------------------------------

STEP 3 — SAFE K-SHAPE CLUSTERING DONE
cluster_input_source: /user/data/kshape/preclustering
full_panel_path: /user/data/kshape/preprocess/full_panel
assignments_csv_out: /workspace/code/kshape/clustering/assignments.csv
panel_with_cluster_out: /user/data/kshape/clustering
matrix_shape: [154, 4067]
n_clusters: 3
random_state: 42
n_locations: 154
n_time_bins: 4067
matrix_cells: 626318
base_bin_minutes: 30
effective_bin_minutes: 450
estimated_kshape_work_gb: 0.1232
was_resampled: True

Cluster counts:
0: 12
1: 58
2: 84


{'cluster_input_source': '/user/data/kshape/preclustering',
 'full_panel_path': '/user/data/kshape/preprocess/full_panel',
 'assignments_csv_out': '/workspace/code/kshape/clustering/assignments.csv',
 'panel_with_cluster_out': '/user/data/kshape/clustering',
 'matrix_shape': [154, 4067],
 'n_clusters': 3,
 'random_state': 42,
 'n_locations': 154,
 'n_time_bins': 4067,
 'matrix_cells': 626318,
 'base_bin_minutes': 30,
 'effective_bin_minutes': 450,
 'estimated_kshape_work_gb': 0.1232,
 'was_resampled': True}